In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
payment_df = spark.read.format("delta").table("dev.tracking_sl.paymentdetail")

In [0]:
dimdate_df = spark.read.format("delta").table("dev.tracking_gd.dimdate")

In [0]:
payment_df = payment_df.select(col('refno').alias('payment_ref'),col('chequeno'),col('details'),col('paymentdate'))

In [0]:
paymentdetail_df = payment_df.join(dimdate_df, to_date(payment_df.paymentdate) == dimdate_df['date'], "inner")

In [0]:
paymentdetail_df = paymentdetail_df.select('payment_ref','datekey','chequeno','details')\
    .withColumn('ingestion_time', current_timestamp())

In [0]:
paymentdetail_df.createOrReplaceTempView("temp_paymentdetail")

In [0]:
%sql
merge into dev.tracking_gd.dimpayment tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_paymentdetail 
) src 
on tgt.payment_ref = src.payment_ref
when not matched then
  insert (payment_ref, datekey, chequeno,details, ingestion_time)
  Values(src.payment_ref,src.datekey,src.chequeno,src.details,src.ingestion_time)

In [0]:
%sql
select * From dev.tracking_gd.dimpayment